# South Korea's Growth and Development analysis

In [ ]:

!pip install statsmodels
!pip install numpy
!pip install pandas

In [ ]:
!pip install matplotlib
!pip install seaborn

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
!pip install openpyxl

In [ ]:
data = pd.read_excel(r'C:\Users\dell\Downloads\pwt_kr_solow_final.xlsx')

In [ ]:
data

In [ ]:
data.index = pd.PeriodIndex(data['year'], freq='Y')
data.drop(columns=['year'], inplace=True)


In [ ]:
data.sort_index(inplace=True)


In [ ]:
data

In [ ]:
data.info()

In [ ]:
print(data.columns, data.size, data.shape, data.ndim)

### create log and lag variables 
### growth rate (g)

In [ ]:
data['ln_gdp'] = np.log(data['real_gdp'])
data['ln_capital'] = np.log(data['capital_stock'])
data['ln_labour'] = np.log(data['labour_force'])
data['ln_investment'] = np.log(data['investment'])
data['ln_consumption'] = np.log(data['consumption'])

# lag 
data['ln_gdp_t-1'] = data['ln_gdp'].shift(1)
data['ln_labour_t-1'] = data['ln_labour'].shift(1)
data['ln_investment_t-1'] = data['investment'].shift(1)
data['ln_consumption_t-1'] = data['consumption'].shift(1)
data['ln_investment_t-2'] = data['investment'].shift(2)
data['ln_consumption_t-2'] = data['consumption'].shift(2)
data['HCI_t-1'] = data['human_capital_index'].shift(1)
data['real_gdp_t-1']= data['real_gdp'].shift(1)

# growth_rate 

data['growth_rate'] = (data['ln_gdp']- data['ln_gdp_t-1'])

In [ ]:
print(data['ln_gdp'],data['ln_capital'],data['ln_labour'])

compute alpha and beta for production function

In [ ]:
# regression 
pro_x = data[['ln_capital','ln_labour']]
# add intercept
pro_x = sm.add_constant(pro_x)
pro_y = data['ln_gdp']

pro_model = sm.OLS(pro_y, pro_x, missing='drop').fit()
print(pro_model.summary())
alpha = pro_model.params['ln_capital']
beta = pro_model.params['ln_labour']
A = np.exp(pro_model.params['const'])

print(f"Calculated Alpha (Capital Elasticity): {alpha:.4f}")
print(f"Calculated Beta (Labor Elasticity): {beta:.4f}")
print(f"Total Factor Productivity (A): {A:.4f}")


In [ ]:
# define Cobb-Douglas production function

def cobb_douglus_fn(A=1.0,alpha = 0.2446):
    return A * (data['capital_stock'] ** alpha ) * (data['labour_force'] **(1 - alpha))

predicted_Y = cobb_douglus_fn()
print (predicted_Y)
    

In [ ]:
# production function graph 
fig, ax = plt.subplots(figsize=(6, 6))

ax.scatter(data['real_gdp'],predicted_Y,color='blue', label='Actual vs Fitted')
limits = [min(min(data['real_gdp']), min(predicted_Y)), max(max(data['real_gdp']), max(predicted_Y))]
ax.plot(limits, limits, color='red', linestyle='--', label='Ideal Fit (y=x)')

ax.set_xlabel('Actual Values')
ax.set_ylabel('Fitted (Predicted) Values')
ax.set_title('Actual vs. Fitted Values Plot')
ax.legend()
ax.grid(True)
plt.show()


In [ ]:
plt.plot(data.index.to_timestamp(), data['ln_consumption'], label = 'consumption')
plt.plot(data.index.to_timestamp(), data['ln_investment'], label = 'investment')
plt.legend()
plt.grid()
plt.show()

In [ ]:
plt.plot(data.index.to_timestamp(), data['ln_gdp'], label = 'gdp')
plt.plot(data.index.to_timestamp(), data['ln_labour'], label = 'labour')
plt.plot(data.index.to_timestamp(), data['real_interest_rate'], label = 'RIR(%)')
plt.plot(data.index.to_timestamp(), data['human_capital_index'], label = 'HCI')
plt.legend()
plt.grid()
plt.show()

# Transitional Dynamics - barro regression

## unconditional regression

In [ ]:
barro_x = data['ln_gdp_t-1']
barro_x = sm.add_constant(barro_x)
barro_y = data['growth_rate']

barro_eqa = sm.OLS(barro_y, barro_x, missing = 'drop').fit()

print(barro_eqa.summary())

convergence_coeff = barro_eqa.params['ln_gdp_t-1']
if convergence_coeff> 0:
    print(f'The economy has divergence growth path with {convergence_coeff:.2f} speed!')
else:
    print(f'The economy has convergence growth path with {convergence_coeff:.2f} speed!')

# conditional regression

In [ ]:
barro_x = data[['ln_gdp_t-1','ln_investment', 'ln_consumption']] # HCI include labour force
barro_x = sm.add_constant(barro_x)
barro_y = data['growth_rate']

barro_eqa = sm.OLS(barro_y, barro_x, missing = 'drop').fit()

print(barro_eqa.summary())

convergence_coeff = barro_eqa.params['ln_gdp_t-1']
if convergence_coeff> 0:
    print(f'The economy has divergence growth path with {convergence_coeff:.2f} speed!')
else:
    print(f'The economy has convergence growth path with {convergence_coeff:.2f} speed!')

In [ ]:
data.corr()

# ARIMA
Autoregressive Integrated Moving Average model

In [ ]:
# random walk
from statsmodels.tsa.arima.model import ARIMA
# Assumes column 'Value' is your time series
ARIMA_model1 = ARIMA(data['ln_gdp'], order=(0,1,0)).fit()
forecast1 = ARIMA_model1.forecast(steps=12)
print(ARIMA_model1.summary())
print(forecast1)

#differenced first-order autoregressive model
ARIMA_model2 = ARIMA(data['ln_gdp'], order=(1,1,0)).fit()
forecast2 = ARIMA_model2.forecast(steps=12)
print(ARIMA_model2.summary())
print(forecast2)

#ARIMA(0,1,1) without constant = simple exponential smoothing
ARIMA_model3 = ARIMA(data['ln_gdp'], order=(0,1,1)).fit()
print(ARIMA_model3.summary())
forecast3 = ARIMA_model3.forecast(steps=12)
print(forecast3)

#ARIMA(1,1,1)
ARIMA_model4 = ARIMA(data['ln_gdp'], order=(1,1,1)).fit()
print(ARIMA_model4.summary())
forecast4 = ARIMA_model4.forecast(steps=12)
print(forecast4)

## per capita terms

In [ ]:
y = (data['real_gdp']/data['labour_force'])
k = (data['capital_stock']/data['labour_force'])
c = (data['consumption']/data['labour_force'])